| Decision | Value |
|---|---|
| Model | `answerdotai/ModernBERT-base` |
| Model type | Encoder-only (BERT-style), MLM head |
| Method | Zero-shot fill-mask scoring |
| Input | `full_prompt` primary, `base_utterance` ablation |
| Labels | ironic=1, non-ironic=0 |
| Scoring | `logit("yes") − logit("no")` at `[MASK]` position |
| Prompt suffix | `"\nAnswer: [MASK]"` appended to every input |
| Positive token | `"yes"` → ironic |
| Negative token | `"no"` → non-ironic |
| Decision rule | `yes_logit > no_logit` → ironic, else non-ironic |
| Training | ❌ None — no gradient updates |
| Split | ❌ None — whole dataset evaluated |
| Evaluation | Separate per condition |
| Primary metric | Macro F1 |
| Secondary metric | Accuracy |
| Per-level metric | Macro F1 per `low` / `high` level |
| Batch size | 16 (no gradients → larger batches fine) |
| Max length | 256 tokens |
| Random seed | 42 |
| Tokenizer | `AutoTokenizer` from `answerdotai/ModernBERT-base` |
| Min transformers version | `>= 4.48.0` |
| Output file | `results_zeroshot.txt` |

## Results Summary

### Overall Performance

| Condition | Macro F1 | Accuracy |
|---|---|---|
| C1 Context Richness | 0.31 | 0.33 |
| C2 Common Ground | 0.42 | 0.52 |
| C3 Prompting Style | 0.45 | 0.50 |

**All very low.** Random chance = 0.50 accuracy, 0.50 macro F1. The model is at or below chance level.

---

### Why so poor?

**ModernBERT-base was never trained for irony detection.** The fill-mask method compares `yes/no` tokens but the model has no understanding of how these words relate to irony. Predictions are essentially random.

---

### Most striking finding — C1 High level

| Level | Macro F1 |
|---|---|
| C1 low | 0.44 |
| C1 high | **0.15** ← catastrophic |

Richer high-context prompts confuse the model even further. It cannot handle longer contextual information.

---

### Pattern visible in per-item predictions

The model almost always predicts `non-ironic`. Ironic and non-ironic versions of the same utterance receive **identical scores** — because the fill-mask approach only sees the prompt structure, not the actual meaning of the utterance.

---

### Conclusion

These results are **expected and actually good news** — because a weak zero-shot baseline makes the improvement from fine-tuning much more meaningful and visible. Once you run the fine-tuning notebook, the comparison will be very compelling.